In [4]:
# === Compare OpenShape pack features vs your <id>.text.npy['text_global'] ===
import os, json, yaml, random, numpy as np
from types import SimpleNamespace as SNS

# ---- SET THIS ----
CONFIG_PATH = "/home/souhailhadgi/OpenShape_code/src/configs/train_ours.yaml"      # path to your training config (yaml or json)
TEXT_NPY_ROOT = "/mnt/disk2/souhail_data/tokfeat_text_npy"   # where <id>.text.npy live
N_SAMPLES = 25                                  # how many random items to inspect
RNG_SEED = 123                                  # reproducibility for coin flips etc.

# ------------- config loader -------------
def _to_ns(x):
    if isinstance(x, dict):
        return SNS(**{k:_to_ns(v) for k,v in x.items()})
    if isinstance(x, list):
        return [_to_ns(v) for v in x]
    return x

def load_config(path):
    with open(path, "r") as f:
        if path.endswith((".yml",".yaml")):
            cfg = yaml.safe_load(f)
        else:
            cfg = json.load(f)
    cfg = _to_ns(cfg)
    # defaults used by training
    if not hasattr(cfg, "dataset"):
        cfg.dataset = SNS()
    ds = cfg.dataset
    ds.use_prompt_engineering = bool(getattr(ds, "use_prompt_engineering", False))
    ds.text_source = list(getattr(ds, "text_source", ["text","caption","retrieval_text"]))
    ds.use_text_filtering = bool(getattr(ds, "use_text_filtering", False))
    ds.gpt4_filtering_path = getattr(ds, "gpt4_filtering_path", None)
    return cfg

config = load_config(CONFIG_PATH)
text_sources = list(config.dataset.text_source)
text_embed_version = "prompt_avg" if config.dataset.use_prompt_engineering else "original"
print("Loaded:", CONFIG_PATH)
print("text_sources:", text_sources, "| text_embed_version:", text_embed_version)

# ------------- helpers: paths & filtering -------------
def pack_path_from_meta(meta):
    # Four rewrites ".../data/..." to "/mnt/disk1/souhail_data/..." for the pack path
    # (matches your repo logic)
    dp = meta.get("data_path", "")
    if "data/" in dp:
        return "/mnt/disk1/souhail_data/" + dp.split("data/")[1]
    return dp

def textnpy_path(meta, root=TEXT_NPY_ROOT):
    dsname = (meta.get("dataset") or "").strip()
    grp    = (meta.get("group")   or "").strip()
    uid    =  meta.get("id") or meta.get("uid") or ""
    base = os.path.join(root, dsname)
    # 3D-FUTURE has no 'group' folder
    if dsname != "3D-FUTURE" and grp:
        base = os.path.join(base, grp)
    return os.path.join(base, f"{uid}.text.npy")

def load_gpt4_filter(cfg):
    if not (cfg.dataset.use_text_filtering and cfg.dataset.gpt4_filtering_path and os.path.exists(cfg.dataset.gpt4_filtering_path)):
        return None
    with open(cfg.dataset.gpt4_filtering_path, "r") as f:
        return json.load(f)

gpt4 = load_gpt4_filter(config)

def gpt4_allows(meta):
    if not gpt4 or not config.dataset.use_text_filtering:
        return True
    uid = meta.get("id") or meta.get("uid") or ""
    ds  = meta.get("dataset") or ""
    entry = gpt4.get(uid) or gpt4.get(f"{ds}/{uid}")
    if not isinstance(entry, dict):
        return True
    return str(entry.get("flag", "Y")).upper() != "N"

# fixed caption order that your <id>.text.npy['text_global'] uses
def texts_fixed_order(pack, include_original=True, include_blip=True, include_msft=True, include_retrieval=True):
    texts, types = [], []
    if include_original:
        t = pack.get("text")
        if isinstance(t, (list, tuple)) and t and isinstance(t[0], str) and t[0]:
            texts.append(t[0]); types.append("original")
    if include_blip:
        t = pack.get("blip_caption")
        if isinstance(t, str) and t:
            texts.append(t); types.append("blip")
    if include_msft:
        t = pack.get("msft_caption")
        if isinstance(t, str) and t:
            texts.append(t); types.append("msft")
    if include_retrieval:
        r = pack.get("retrieval_text")
        if isinstance(r, (list, tuple)):
            for s in r:
                if isinstance(s, str) and s:
                    texts.append(s); types.append("retrieval")
    return texts, types

def cosine_sim(a, b, eps=1e-8):
    a = a.astype(np.float32); b = b.astype(np.float32)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + eps))

# ------------- OpenShape-style chooser (matches Four) -------------
rng = np.random.RandomState(RNG_SEED)

def choose_openshape_caption_and_feat(pack, meta):
    """
    Build candidates per enabled sources, exactly like Four:
      - Objaverse 'text' uses text[0] if GPT-4 allows; others pick random idx in text list
      - 'caption' coin-flips BLIP/MSFT if present
      - 'retrieval_text' picks a random retrieval
    Then take 1 random from the candidate list (if non-empty).
    Returns dict with: source_tag ('original'|'blip'|'msft'|'retrieval' or 'text[idx]'),
    chosen_text (str), chosen_feat (np.ndarray), retrieval_idx (int or None), text_idx (int or None)
    """
    dsname = meta.get("dataset", "")
    cands = []

    # text
    if "text" in text_sources:
        if dsname == "Objaverse":
            if gpt4_allows(meta) and "text" in pack and "text_feat" in pack and len(pack["text"]) > 0:
                i = 0
                tf = pack["text_feat"][i].get(text_embed_version, None)
                if tf is not None:
                    cands.append(("original", pack["text"][i], np.asarray(tf), None, i))
        else:
            if "text" in pack and "text_feat" in pack and len(pack["text"]) > 0:
                i = rng.randint(len(pack["text"]))
                tf = pack["text_feat"][i].get(text_embed_version, None)
                if tf is not None:
                    cands.append((f"text[{i}]", pack["text"][i], np.asarray(tf), None, i))

    # caption (coin flip BLIP / MSFT)
    if "caption" in text_sources:
        if rng.rand() < 0.5:
            if "blip_caption" in pack and pack.get("blip_caption") and "blip_caption_feat" in pack:
                tf = pack["blip_caption_feat"].get(text_embed_version, None)
                if tf is not None:
                    cands.append(("blip", pack["blip_caption"], np.asarray(tf), None, None))
        else:
            if "msft_caption" in pack and pack.get("msft_caption") and "msft_caption_feat" in pack:
                tf = pack["msft_caption_feat"].get(text_embed_version, None)
                if tf is not None:
                    cands.append(("msft", pack["msft_caption"], np.asarray(tf), None, None))

    # retrieval_text
    if "retrieval_text" in text_sources and "retrieval_text" in pack and "retrieval_text_feat" in pack:
        if len(pack["retrieval_text"]) > 0:
            i = rng.randint(len(pack["retrieval_text"]))
            tf = pack["retrieval_text_feat"][i].get("original", None)
            if tf is not None:
                cands.append(("retrieval", pack["retrieval_text"][i], np.asarray(tf), i, None))

    if not cands:
        return None

    tag, txt, feat, ridx, tidx = cands[rng.randint(len(cands))]
    return {"tag": tag, "text": txt, "feat": feat.astype(np.float32).reshape(-1), "retrieval_idx": ridx, "text_idx": tidx}

# ------------- runner over split -------------
# Load split
with open(config.dataset.train_split, "r") as f:
    split = json.load(f)
idxs = list(range(len(split)))
rng.shuffle(idxs)
idxs = idxs[:N_SAMPLES]

mismatches = []
skipped_text_random = 0
print(f"Scanning {len(idxs)} items out of {len(split)}…\n")

for k, i in enumerate(idxs, 1):
    meta = split[i]
    dsname = meta.get("dataset","")
    gid    = meta.get("group","")
    uid    = meta.get("id","")
    ppack  = pack_path_from_meta(meta)
    ptext  = textnpy_path(meta)

    # load OpenShape pack
    if not (ppack and os.path.exists(ppack)):
        print(f"[{k}] {dsname}/{gid}/{uid}  pack MISSING: {ppack}")
        continue
    try:
        pack = np.load(ppack, allow_pickle=True).item()
    except Exception as e:
        print(f"[{k}] {dsname}/{gid}/{uid}  pack LOAD ERROR: {ppack} -> {repr(e)}")
        continue

    # choose caption & vector the OpenShape way
    choice = choose_openshape_caption_and_feat(pack, meta)
    if choice is None:
        print(f"[{k}] {dsname}/{gid}/{uid}  no OS text candidate (keys missing or filtered).")
        continue

    os_tag  = choice["tag"]          # 'original'|'blip'|'msft'|'retrieval' or 'text[i]'
    os_text = choice["text"]
    os_vec  = choice["feat"]

    # compute index into your text_global according to fixed ordering
    ordered_texts, ordered_types = texts_fixed_order(pack)
    # For 'text[i]' on non-Objaverse, only i==0 maps to 'original' row; others aren't in text_global
    map_idx = None
    if os_tag == "original":
        if "original" in ordered_types:
            map_idx = ordered_types.index("original")
    elif os_tag == "blip":
        if "blip" in ordered_types:
            map_idx = ordered_types.index("blip")
    elif os_tag == "msft":
        if "msft" in ordered_types:
            map_idx = ordered_types.index("msft")
    elif os_tag == "retrieval":
        ridxs = [j for j,t in enumerate(ordered_types) if t == "retrieval"]
        if ridxs and choice["retrieval_idx"] is not None and choice["retrieval_idx"] < len(ridxs):
            map_idx = ridxs[choice["retrieval_idx"]]
    elif os_tag.startswith("text["):
        # non-Objaverse 'text' picked random i; only i==0 is saved as 'original' in text_global
        tidx = choice["text_idx"]
        if tidx == 0 and "original" in ordered_types:
            map_idx = ordered_types.index("original")
        else:
            skipped_text_random += 1

    # load your <id>.text.npy and pull text_global
    if not (ptext and os.path.exists(ptext)):
        print(f"[{k}] {dsname}/{gid}/{uid}  <id>.text.npy MISSING: {ptext}")
        continue
    try:
        obj = np.load(ptext, allow_pickle=True)
        if isinstance(obj, np.ndarray) and obj.dtype == object and obj.ndim == 0:
            obj = obj.item()
        arr = obj.get("text_global", None) if isinstance(obj, dict) else obj
        if arr is None:
            print(f"[{k}] {dsname}/{gid}/{uid}  'text_global' not found in {ptext}")
            continue
        arr = np.asarray(arr, dtype=np.float32)
        if arr.ndim == 1:
            arr = arr[None, :]
    except Exception as e:
        print(f"[{k}] {dsname}/{gid}/{uid}  <id>.text.npy LOAD ERROR: {ptext} -> {repr(e)}")
        continue

    # sanity: length alignment
    if arr.shape[0] != len(ordered_types):
        print(f"[{k}] {dsname}/{gid}/{uid}  ROW MISMATCH: text_global rows={arr.shape[0]} vs ordered_types={len(ordered_types)}")

    if map_idx is None:
        print(f"[{k}] {dsname}/{gid}/{uid}] chose '{os_tag}' → no direct row in text_global (likely non-Obj 'text[i>0]').")
        continue
    if map_idx >= arr.shape[0]:
        print(f"[{k}] {dsname}/{gid}/{uid}  map_idx {map_idx} out of range rows={arr.shape[0]}")
        continue

    npy_vec = arr[map_idx].reshape(-1)
    if npy_vec.shape != os_vec.shape:
        print(f"[{k}] {dsname}/{gid}/{uid}  DIM MISMATCH: OS {os_vec.shape} vs NPY {npy_vec.shape}  (tag={os_tag})")
        continue

    l2 = float(np.linalg.norm(os_vec - npy_vec))
    cs = cosine_sim(os_vec, npy_vec)
    mx = float(np.max(np.abs(os_vec - npy_vec)))
    print(f"[{k}] {dsname}/{gid}/{uid}  tag={os_tag:9s}  L2={l2:.6f}  cos={cs:.6f}  max|diff|={mx:.6f}"
          f"\n     pack={ppack}\n     npy ={ptext}")

    if not (np.isfinite(l2) and l2 < 1e-4 and cs > 0.9999 and mx < 1e-5):
        mismatches.append((dsname, gid, uid, os_tag, l2, cs, mx))

print("\nDone.")
print("Skipped comparisons due to non-Obj 'text[i>0]':", skipped_text_random)
print(f"Mismatches (loose threshold) count: {len(mismatches)}")
if mismatches[:5]:
    print("First few mismatches:", mismatches[:5])


Loaded: /home/souhailhadgi/OpenShape_code/src/configs/train_ours.yaml
text_sources: ['text', 'caption', 'retrieval_text'] | text_embed_version: original
Scanning 25 items out of 829460…

[1] Objaverse/000-104/53a02225e80a4bf2992d7f8072781aab  tag=original   L2=39.402798  cos=0.285606  max|diff|=4.619067
     pack=/mnt/disk1/souhail_data/objaverse-processed/merged_for_training_final/Objaverse/000-104/53a02225e80a4bf2992d7f8072781aab.npy
     npy =/mnt/disk2/souhail_data/tokfeat_text_npy/Objaverse/000-104/53a02225e80a4bf2992d7f8072781aab.text.npy
[2] ShapeNet/04379243/14f3de4c23d535e92c528d33bca1ac2  tag=msft       L2=36.508274  cos=0.376533  max|diff|=3.557241
     pack=/mnt/disk1/souhail_data/objaverse-processed/merged_for_training_final/ShapeNet/04379243/14f3de4c23d535e92c528d33bca1ac2.npy
     npy =/mnt/disk2/souhail_data/tokfeat_text_npy/ShapeNet/04379243/14f3de4c23d535e92c528d33bca1ac2.text.npy
[3] Objaverse/000-156/489363566377492c84b08dd0c83778e0  tag=blip       L2=40.261410  cos

In [ ]:
# === Compare 4 ways: PACK vs NPY vs API (encode_text) vs MANUAL (CLIP forward) ===
import os, json, yaml, random, warnings
import numpy as np
import torch
import pandas as pd
from types import SimpleNamespace as SNS

# --------- CONFIGURE THESE ---------
CONFIG_PATH     = "/home/souhailhadgi/OpenShape_code/src/configs/train_ours.yaml"      # <-- your config (yaml/json)
TEXT_NPY_ROOT   = "/mnt/disk2/souhail_data/tokfeat_text_npy"
MODEL_NAME      = "ViT-bigG-14"
PRETRAINED      = "laion2b_s39b_b160k"             # (if not using local .pt)
PRETRAINED_PATH = None                              # e.g. "/path/to/bigG-laion2b_s39b_b160k.pt"
N_ITEMS         = 12                                # sample size
RNG_SEED        = 123
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

# ---------- helpers ----------
def _to_ns(x):
    if isinstance(x, dict): return SNS(**{k:_to_ns(v) for k,v in x.items()})
    if isinstance(x, list): return [_to_ns(v) for v in x]
    return x

def load_config(path):
    with open(path, "r") as f:
        cfg = yaml.safe_load(f) if path.endswith((".yml",".yaml")) else json.load(f)
    cfg = _to_ns(cfg)
    if not hasattr(cfg, "dataset"): cfg.dataset = SNS()
    ds = cfg.dataset
    ds.train_split = getattr(ds, "train_split", None)
    ds.use_prompt_engineering = bool(getattr(ds, "use_prompt_engineering", False))
    return cfg

def pack_path_from_meta(meta):
    dp = meta.get("data_path", "")
    if "data/" in dp:
        return "/mnt/disk1/souhail_data/" + dp.split("data/")[1]
    return dp or None

def textnpy_path(meta, root=TEXT_NPY_ROOT):
    dsname = (meta.get("dataset") or "").strip()
    grp    = (meta.get("group")   or "").strip()
    uid    =  meta.get("id") or meta.get("uid") or ""
    base = os.path.join(root, dsname)
    if dsname != "3D-FUTURE" and grp:
        base = os.path.join(base, grp)
    return os.path.join(base, f"{uid}.text.npy")

def texts_fixed_order(pack, include_original=True, include_blip=True, include_msft=True, include_retrieval=True):
    texts, types = [], []
    if include_original:
        t = pack.get("text")
        if isinstance(t, (list, tuple)) and t and isinstance(t[0], str) and t[0]:
            texts.append(t[0]); types.append("original")
    if include_blip:
        t = pack.get("blip_caption")
        if isinstance(t, str) and t:
            texts.append(t); types.append("blip")
    if include_msft:
        t = pack.get("msft_caption")
        if isinstance(t, str) and t:
            texts.append(t); types.append("msft")
    if include_retrieval:
        r = pack.get("retrieval_text")
        if isinstance(r, (list, tuple)):
            for s in r:
                if isinstance(s, str) and s:
                    texts.append(s); types.append("retrieval")
    return texts, types

def cosine_sim(a, b, eps=1e-8):
    a = a.astype(np.float32); b = b.astype(np.float32)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + eps))

# ---------- load config + split ----------
config = load_config(CONFIG_PATH)
assert config.dataset.train_split and os.path.exists(config.dataset.train_split), f"Can't find split: {config.dataset.train_split}"
with open(config.dataset.train_split, "r") as f:
    split = json.load(f)

text_embed_version = "prompt_avg" if config.dataset.use_prompt_engineering else "original"
print("Loaded config.")
print("text_embed_version:", text_embed_version)

# ---------- load OpenCLIP ----------
try:
    import open_clip
except Exception as e:
    raise RuntimeError("Need `open_clip` (pip install open_clip_torch).") from e

print(f"Loading {MODEL_NAME} / {PRETRAINED_PATH or PRETRAINED} on {DEVICE} …")
if PRETRAINED_PATH and os.path.exists(PRETRAINED_PATH):
    model, _, _ = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED_PATH, device=DEVICE)
else:
    model, _, _ = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED, device=DEVICE)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.eval().to(DEVICE)
print("context_length:", getattr(model, "context_length", None))

@torch.no_grad()
def encode_text_manual(model, token_ids: torch.Tensor) -> torch.Tensor:
    """Manual CLIP text forward (global at EOT). Returns (N,D) normalized float32."""
    x = model.token_embedding(token_ids)                 # (N, L, C)
    x = x + model.positional_embedding.to(x.dtype)
    x = x.permute(1, 0, 2)                               # (L, N, C)
    x = model.transformer(x)
    x = x.permute(1, 0, 2)                               # (N, L, C)
    x = model.ln_final(x)
    if getattr(model, "text_projection", None) is not None:
        x = x @ model.text_projection
    x = x.float()
    eot_pos = token_ids.argmax(dim=-1)                   # (N,)
    glo = x[torch.arange(x.shape[0]), eot_pos, :]
    glo = torch.nn.functional.normalize(glo, dim=-1)
    return glo

@torch.no_grad()
def encode_text_api(model, token_ids: torch.Tensor) -> torch.Tensor:
    feats = model.encode_text(token_ids)
    feats = torch.nn.functional.normalize(feats.float(), dim=-1)
    return feats

# ---------- sample indices ----------
rng = random.Random(RNG_SEED)
indices = list(range(len(split)))
rng.shuffle(indices)
indices = indices[:N_ITEMS]

rows = []
print(f"\nComparing on {len(indices)} items…\n")

for k, idx in enumerate(indices, 1):
    meta = split[idx]
    dsname = meta.get("dataset","")
    gid    = meta.get("group","")
    uid    = meta.get("id","")
    ppack  = pack_path_from_meta(meta)
    ptext  = textnpy_path(meta)

    # load pack
    if not (ppack and os.path.exists(ppack)):
        print(f"[{k}] {dsname}/{gid}/{uid}  PACK missing -> {ppack}")
        continue
    try:
        pack = np.load(ppack, allow_pickle=True).item()
    except Exception as e:
        print(f"[{k}] {dsname}/{gid}/{uid}  PACK load error: {repr(e)}")
        continue

    # ordered captions (what text_global aligns to)
    ordered_texts, ordered_types = texts_fixed_order(pack)
    T_order = len(ordered_texts)

    # load <id>.text.npy -> dict['text_global']
    if not (ptext and os.path.exists(ptext)):
        print(f"[{k}] {dsname}/{gid}/{uid}  NPY missing -> {ptext}")
        continue
    try:
        obj = np.load(ptext, allow_pickle=True)
        if isinstance(obj, np.ndarray) and obj.dtype == object and obj.ndim == 0:
            obj = obj.item()
        tg = obj.get("text_global", None) if isinstance(obj, dict) else obj
        if tg is None:
            print(f"[{k}] {dsname}/{gid}/{uid}  'text_global' not in NPY")
            continue
        tg = np.asarray(tg, dtype=np.float32)
        if tg.ndim == 1: tg = tg[0:1]
    except Exception as e:
        print(f"[{k}] {dsname}/{gid}/{uid}  NPY load error: {repr(e)}")
        continue

    T = min(T_order, tg.shape[0])
    if T == 0:
        print(f"[{k}] {dsname}/{gid}/{uid}  no comparable rows (T==0)")
        continue

    # Build PACK features aligned per row/type (deterministic, no coin-flip)
    pack_vecs = [None]*T
    # precompute retrieval row indices within ordered list
    retrieval_row_indices = [j for j,t in enumerate(ordered_types[:T]) if t == "retrieval"]
    for j in range(T):
        typ = ordered_types[j]
        try:
            if typ == "original":
                # map to text[0] feature with chosen version
                if "text_feat" in pack and isinstance(pack["text_feat"], (list, tuple)) and len(pack["text_feat"]) > 0:
                    vec = pack["text_feat"][0].get(text_embed_version, None)
                    pack_vecs[j] = np.asarray(vec, dtype=np.float32).reshape(-1) if vec is not None else None
            elif typ == "blip":
                if "blip_caption_feat" in pack and isinstance(pack["blip_caption_feat"], dict):
                    vec = pack["blip_caption_feat"].get(text_embed_version, None)
                    pack_vecs[j] = np.asarray(vec, dtype=np.float32).reshape(-1) if vec is not None else None
            elif typ == "msft":
                if "msft_caption_feat" in pack and isinstance(pack["msft_caption_feat"], dict):
                    vec = pack["msft_caption_feat"].get(text_embed_version, None)
                    pack_vecs[j] = np.asarray(vec, dtype=np.float32).reshape(-1) if vec is not None else None
            elif typ == "retrieval":
                # which retrieval is this row? use position among retrieval rows
                r_pos = retrieval_row_indices.index(j)  # 0..R-1
                if "retrieval_text_feat" in pack and len(pack["retrieval_text_feat"]) > r_pos:
                    # retrieval feats are always 'original' (no prompt eng)
                    vec = pack["retrieval_text_feat"][r_pos].get("original", None)
                    pack_vecs[j] = np.asarray(vec, dtype=np.float32).reshape(-1) if vec is not None else None
        except Exception:
            pack_vecs[j] = None

    # encode captions (API & MANUAL) in a batch
    caps = ordered_texts[:T]
    toks = tokenizer(caps).to(DEVICE)
    with torch.no_grad():
        api_feats = encode_text_api(model, toks).cpu().numpy()      # (T, D)
        man_feats = encode_text_manual(model, toks).cpu().numpy()   # (T, D)

    # compare pairs for each row
    for j in range(T):
        typ = ordered_types[j]
        cap = caps[j]
        vec_pack = pack_vecs[j]
        vec_npy  = tg[j].reshape(-1)
        vec_api  = api_feats[j].reshape(-1)
        vec_man  = man_feats[j].reshape(-1)

        def safe_cos(a, b):
            return None if (a is None or b is None) else cosine_sim(a, b)
        def safe_l2(a, b):
            return None if (a is None or b is None) else float(np.linalg.norm(a - b))
        def safe_max(a, b):
            return None if (a is None or b is None) else float(np.max(np.abs(a - b)))

        rows.append({
            "dataset": dsname, "group": gid, "id": uid, "row": j, "type": typ,
            "cos(pack,npy)": safe_cos(vec_pack, vec_npy),
            "L2(pack,npy)" : safe_l2(vec_pack, vec_npy),
            "max|p-n|"     : safe_max(vec_pack, vec_npy),
            "cos(pack,api)": safe_cos(vec_pack, vec_api),
            "cos(pack,man)": safe_cos(vec_pack, vec_man),
            "cos(api,npy)" : cosine_sim(vec_api, vec_npy),
            "L2(api,npy)"  : float(np.linalg.norm(vec_api - vec_npy)),
            "max|a-n|"     : float(np.max(np.abs(vec_api - vec_npy))),
            "cos(man,npy)" : cosine_sim(vec_man, vec_npy),
            "L2(man,npy)"  : float(np.linalg.norm(vec_man - vec_npy)),
            "max|m-n|"     : float(np.max(np.abs(vec_man - vec_npy))),
            "cos(man,api)" : cosine_sim(vec_man, vec_api),
            "pack_has_vec" : vec_pack is not None,
            "npy_path": ptext, "pack_path": ppack,
            "caption": cap if len(cap) <= 100 else (cap[:100] + "…"),
        })

# ---------- summarize ----------
df = pd.DataFrame(rows)
if df.empty:
    print("\nNo rows to compare (check paths/config).")
else:
    # Reasonable tight thresholds (tune if needed)
    COS_GOOD = 0.9999
    L2_GOOD  = 1e-4
    MAXD_GOOD= 1e-5

    def flag_bad(cosv, l2v, mxv):
        return (cosv is None) or (cosv < COS_GOOD) or (l2v is None) or (l2v > L2_GOOD) or (mxv is None) or (mxv > MAXD_GOOD)

    df["bad_pack_npy"] = df.apply(lambda r: flag_bad(r["cos(pack,npy)"], r["L2(pack,npy)"], r["max|p-n|"]), axis=1)

    print(f"\nTotal comparable rows: {len(df)}")
    print("Rows missing PACK vector (pack_has_vec=False):", int((~df["pack_has_vec"]).sum()))
    print("Suspect mismatches (PACK vs NPY):", int(df["bad_pack_npy"].sum()))

    # Show worst offenders for PACK vs NPY
    df_show = df.copy()
    # Put None as -inf for sorting
    df_show["cos(pack,npy)_s"] = df_show["cos(pack,npy)"].fillna(-1.0)
    worst = df_show.sort_values(by=["cos(pack,npy)_s"]).head(12)
    display(worst[[
        "dataset","group","id","row","type",
        "cos(pack,npy)","L2(pack,npy)","max|p-n|",
        "cos(pack,api)","cos(pack,man)",
        "cos(api,npy)","cos(man,npy)","cos(man,api)",
        "pack_path","npy_path","caption"
    ]])

    # Optional: per-type summary
    print("\nPer-type mean cosine:")b
    with pd.option_context('display.float_format', '{:.6f}'.format):
        display(df.groupby("type")[["cos(pack,npy)","cos(pack,api)","cos(pack,man)","cos(api,npy)","cos(man,npy)","cos(man,api)"]].mean())


Loaded config.
text_embed_version: original
Loading ViT-bigG-14 / laion2b_s39b_b160k on cuda …
context_length: 77

Comparing on 12 items…


Total comparable rows: 203
Rows missing PACK vector (pack_has_vec=False): 0
Suspect mismatches (PACK vs NPY): 203


,dataset,group,id,row,type,"cos(pack,npy)","L2(pack,npy)",max|p-n|,"cos(pack,api)","cos(pack,man)","cos(api,npy)","cos(man,npy)","cos(man,api)",pack_path,npy_path,caption
97,Objaverse,000-061,c6ced88817194f7aade2230795eaa5f8,8,retrieval,0.027480,46.229774,4.648268,1.0,0.008789,0.027480,0.888172,0.008789,/mnt/disk1/souhail_data/objaverse-processed/me...,/mnt/disk2/souhail_data/tokfeat_text_npy/Objav...,Palm Tree Dypsis Lutescens PNG & PSD Images
192,Objaverse,000-041,b8d3eaa9239e47698e4b35a5de6d6016,4,retrieval,0.064203,42.382626,4.523208,1.0,0.083877,0.064203,0.913615,0.083877,/mnt/disk1/souhail_data/objaverse-processed/me...,/mnt/disk2/souhail_data/tokfeat_text_npy/Objav...,Chichen Itza icon in outline style isolated on...
47,Objaverse,000-039,535455b539d0416d9605b0ddc5fb0f0c,9,retrieval,0.076708,43.472248,4.368088,1.0,0.069473,0.076708,0.918051,0.069473,/mnt/disk1/souhail_data/objaverse-processed/me...,/mnt/disk2/souhail_data/tokfeat_text_npy/Objav...,Silhouette Poster A4 21 x 29.7 cm von by Lasse...
86,Objaverse,000-148,1349d05138b844369f62b2e52dcfcb2f,12,retrieval,0.080178,42.849751,3.818098,1.0,0.071711,0.080178,0.894666,0.071711,/mnt/disk1/souhail_data/objaverse-processed/me...,/mnt/disk2/souhail_data/tokfeat_text_npy/Objav...,Titanfall Northstar B003128 file stl free down...
163,Objaverse,000-101,0c6857855c184eaeabe77751ba0f38b9,10,retrieval,0.101184,42.627502,4.530256,1.0,0.069568,0.101184,0.897652,0.069568,/mnt/disk1/souhail_data/objaverse-processed/me...,/mnt/disk2/souhail_data/tokfeat_text_npy/Objav...,Justin Herbert 2020 Donruss Optic Rookie Pheno...
166,Objaverse,000-101,0c6857855c184eaeabe77751ba0f38b9,13,retrieval,0.108444,41.894135,4.071651,1.0,0.091292,0.108444,0.877834,0.091292,/mnt/disk1/souhail_data/objaverse-processed/me...,/mnt/disk2/souhail_data/tokfeat_text_npy/Objav...,Justin Herbert 2020 Leaf Valiant All American ...
168,Objaverse,000-101,0c6857855c184eaeabe77751ba0f38b9,15,retrieval,0.111714,41.639202,4.156409,1.0,0.063801,0.111714,0.878065,0.063801,/mnt/disk1/souhail_data/objaverse-processed/me...,/mnt/disk2/souhail_data/tokfeat_text_npy/Objav...,Justin Herbert 2020 Panini Origins Rookie Auto...
156,Objaverse,000-101,0c6857855c184eaeabe77751ba0f38b9,3,retrieval,0.111888,42.474625,4.483483,1.0,0.113752,0.111888,0.853195,0.113752,/mnt/disk1/souhail_data/objaverse-processed/me...,/mnt/disk2/souhail_data/tokfeat_text_npy/Objav...,2020 Panini Select Justin Herbert Patch Auto S...
102,Objaverse,000-061,c6ced88817194f7aade2230795eaa5f8,13,retrieval,0.113822,44.188618,5.173100,1.0,0.132986,0.113823,0.905760,0.132987,/mnt/disk1/souhail_data/objaverse-processed/me...,/mnt/disk2/souhail_data/tokfeat_text_npy/Objav...,Tree: Palm Trees PNG & PSD Images
100,Objaverse,000-061,c6ced88817194f7aade2230795eaa5f8,11,retrieval,0.114014,44.056187,4.652023,1.0,0.094034,0.114014,0.900441,0.094033,/mnt/disk1/souhail_data/objaverse-processed/me...,/mnt/disk2/souhail_data/tokfeat_text_npy/Objav...,Roystonea: Royal Palm PNG & PSD Images



Per-type mean cosine:


,"cos(pack,npy)","cos(pack,api)","cos(pack,man)","cos(api,npy)","cos(man,npy)","cos(man,api)"
type,,,,,,
blip,0.289907,1.000000,0.310245,0.289907,0.881545,0.310245
msft,0.290094,1.000000,0.299754,0.290094,0.851273,0.299754
original,0.419930,1.000000,0.407393,0.419930,0.804732,0.407393
retrieval,0.298212,1.000000,0.298082,0.298212,0.833160,0.298082


In [3]:
# === Float16 roundtrip + scale diagnosis: are PACK vectors just scaled? ===
import os, json, yaml, random, warnings, numpy as np, torch, pandas as pd
from types import SimpleNamespace as SNS

# ---------- EDIT THESE ----------
CONFIG_PATH     = "/home/souhailhadgi/OpenShape_code/src/configs/train_ours.yaml"
OUT_ROOT        = "/mnt/disk2/souhail_data"   # temp npz save dir
MODEL_NAME      = "ViT-bigG-14"
PRETRAINED      = "laion2b_s39b_b160k"
PRETRAINED_PATH = None                                  # or a local .pt
N_ITEMS         = 12
RNG_SEED        = 123
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

# ---------- helpers ----------
def _to_ns(x):
    if isinstance(x, dict): return SNS(**{k:_to_ns(v) for k,v in x.items()})
    if isinstance(x, list): return [_to_ns(v) for v in x]
    return x

def load_config(p):
    with open(p, "r") as f:
        cfg = yaml.safe_load(f) if p.endswith((".yml",".yaml")) else json.load(f)
    cfg = _to_ns(cfg)
    if not hasattr(cfg, "dataset"): cfg.dataset = SNS()
    return cfg

def pack_path_from_meta(meta):
    dp = meta.get("data_path","")
    return "/mnt/disk1/souhail_data/" + dp.split("data/")[1] if "data/" in dp else dp or None

def texts_fixed_order(pack):
    texts, types = [], []
    t = pack.get("text")
    if isinstance(t, (list, tuple)) and t and isinstance(t[0], str) and t[0]:
        texts.append(t[0]); types.append("original")
    t = pack.get("blip_caption")
    if isinstance(t, str) and t:
        texts.append(t); types.append("blip")
    t = pack.get("msft_caption")
    if isinstance(t, str) and t:
        texts.append(t); types.append("msft")
    r = pack.get("retrieval_text")
    if isinstance(r, (list, tuple)):
        for s in r:
            if isinstance(s, str) and s:
                texts.append(s); types.append("retrieval")
    return texts, types

def cosine_sim(a, b, eps=1e-8):
    a = a.astype(np.float32); b = b.astype(np.float32)
    return float(np.dot(a, b) / (np.linalg.norm(a)*np.linalg.norm(b) + eps))

# ---------- load config + split ----------
config = load_config(CONFIG_PATH)
assert getattr(config.dataset, "train_split", None) and os.path.exists(config.dataset.train_split), "train_split missing"
with open(config.dataset.train_split, "r") as f:
    split = json.load(f)

text_embed_version = "prompt_avg" if getattr(config.dataset, "use_prompt_engineering", False) else "original"
print("text_embed_version:", text_embed_version)

# ---------- load OpenCLIP ----------
import open_clip
print(f"Loading {MODEL_NAME} / {PRETRAINED_PATH or PRETRAINED} on {DEVICE} …")
if PRETRAINED_PATH and os.path.exists(PRETRAINED_PATH):
    model, _, _ = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED_PATH, device=DEVICE)
else:
    model, _, _ = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED, device=DEVICE)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.eval().to(DEVICE)

@torch.no_grad()
def encode_text_api(model, token_ids: torch.Tensor) -> torch.Tensor:
    x = model.encode_text(token_ids)
    return torch.nn.functional.normalize(x.float(), dim=-1)  # unit norm

# ---------- sample items ----------
rng = random.Random(RNG_SEED)
idxs = list(range(len(split)))
rng.shuffle(idxs)
idxs = idxs[:N_ITEMS]

os.makedirs(OUT_ROOT, exist_ok=True)
rows = []
print(f"\nProcessing {len(idxs)} items…\n")

for k, i in enumerate(idxs, 1):
    meta = split[i]
    dsname = meta.get("dataset","")
    gid    = meta.get("group","")
    uid    = meta.get("id","")
    ppack  = pack_path_from_meta(meta)

    if not (ppack and os.path.exists(ppack)):
        print(f"[{k}] {dsname}/{gid}/{uid}  PACK missing -> {ppack}")
        continue
    try:
        pack = np.load(ppack, allow_pickle=True).item()
    except Exception as e:
        print(f"[{k}] {dsname}/{gid}/{uid}  PACK load error: {repr(e)}")
        continue

    caps, types = texts_fixed_order(pack)
    T = len(caps)
    if T == 0: 
        print(f"[{k}] {dsname}/{gid}/{uid}  no captions")
        continue

    toks = tokenizer(caps).to(DEVICE)
    with torch.no_grad():
        api = encode_text_api(model, toks).cpu().numpy()     # (T,D) unit norm float32

    # save/reload npz float16 (embeddings.py style)
    out_dir  = os.path.join(OUT_ROOT, dsname, gid or "_")
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{uid}.text_global.fp16.npz")
    np.savez_compressed(out_path, text_global=api.astype(np.float16))

    npz = np.load(out_path)["text_global"].astype(np.float32)  # (T,D), ~unit norm

    # build aligned PACK vectors per row/type (deterministic mapping)
    pack_vecs = [None]*T
    r_rows = [j for j,t in enumerate(types) if t == "retrieval"]
    for j in range(T):
        typ = types[j]
        try:
            if typ == "original":
                vec = pack["text_feat"][0].get(text_embed_version, None)
            elif typ == "blip":
                vec = pack.get("blip_caption_feat", {}).get(text_embed_version, None)
            elif typ == "msft":
                vec = pack.get("msft_caption_feat", {}).get(text_embed_version, None)
            else:  # retrieval
                rpos = r_rows.index(j)
                vec = pack["retrieval_text_feat"][rpos].get("original", None)
            pack_vecs[j] = None if vec is None else np.asarray(vec, dtype=np.float32).reshape(-1)
        except Exception:
            pack_vecs[j] = None

    # compare raw + diagnose scale
    for j in range(T):
        cap  = caps[j]; typ = types[j]
        v_npz = npz[j];           # unit-ish
        v_pck = pack_vecs[j]
        if v_pck is None:
            rows.append({
                "dataset": dsname, "group": gid, "id": uid, "row": j, "type": typ,
                "cos(NPZ,PACK)": None, "L2(NPZ,PACK)": None, "max|N-P|": None,
                "pack_norm": None, "s_best": None,
                "L2(PACK, s*NPZ)": None, "cos(NPZ, PACK_norm)": None,
                "pack_path": ppack, "npz_path": out_path, "caption": cap[:100]+"…" if len(cap)>100 else cap,
            })
            continue

        # raw
        cos_raw = cosine_sim(v_npz, v_pck)
        l2_raw  = float(np.linalg.norm(v_npz - v_pck))
        mx_raw  = float(np.max(np.abs(v_npz - v_pck)))

        # scale diagnostics
        pn = float(np.linalg.norm(v_pck))
        # best scalar s that minimizes ||v_pck - s*v_npz|| = (v_npz·v_pck)/(v_npz·v_npz)
        s_best = float(np.dot(v_npz, v_pck) / max(1e-8, np.dot(v_npz, v_npz)))
        l2_rescaled = float(np.linalg.norm(v_pck - s_best * v_npz))

        # normalized comparison (unit norms on both)
        v_pck_n = v_pck / (pn + 1e-8)
        cos_unit = cosine_sim(v_npz, v_pck_n)

        rows.append({
            "dataset": dsname, "group": gid, "id": uid, "row": j, "type": typ,
            "cos(NPZ,PACK)": cos_raw, "L2(NPZ,PACK)": l2_raw, "max|N-P|": mx_raw,
            "pack_norm": pn, "s_best": s_best,
            "L2(PACK, s*NPZ)": l2_rescaled, "cos(NPZ, PACK_norm)": cos_unit,
            "pack_path": ppack, "npz_path": out_path, "caption": cap[:100]+"…" if len(cap)>100 else cap,
        })

df = pd.DataFrame(rows)
if df.empty:
    print("No rows compared. Check paths/config.")
else:
    with pd.option_context('display.float_format', '{:.6f}'.format):
        print("\n=== Raw (NPZ16 vs PACK) — scale mismatch will inflate L2 ===")
        display(df[["cos(NPZ,PACK)","L2(NPZ,PACK)","max|N-P|"]].describe())

        print("\n=== Pack norms and best scale s (PACK ≈ s * NPZ) ===")
        display(df[["pack_norm","s_best"]].describe())

        print("\n=== After compensation ===")
        print("• L2(PACK, s*NPZ)  -> should be ~0 if only a scale differs")
        print("• cos(NPZ, PACK_norm) -> should be ~1 if directions match")
        display(df[["L2(PACK, s*NPZ)","cos(NPZ, PACK_norm)"]].describe())

        # worst by cos after normalization (should still be ~1 if ok)
        dfx = df.copy()
        dfx["cos_sort"] = dfx["cos(NPZ, PACK_norm)"].fillna(-1.0)
        print("\nWorst 10 by cos after normalization:")
        display(dfx.sort_values("cos_sort").head(10)[
            ["dataset","group","id","row","type","cos(NPZ, PACK_norm)","L2(PACK, s*NPZ)","pack_norm","s_best","pack_path","npz_path","caption"]
        ])


text_embed_version: original
Loading ViT-bigG-14 / laion2b_s39b_b160k on cuda …

Processing 12 items…


=== Raw (NPZ16 vs PACK) — scale mismatch will inflate L2 ===


,"cos(NPZ,PACK)","L2(NPZ,PACK)",max|N-P|
count,203.000000,203.000000,203.000000
mean,1.000000,34.824992,4.210701
std,0.000000,2.414184,0.723030
min,1.000000,26.797806,2.691619
25%,1.000000,33.458366,3.684318
50%,1.000000,34.591110,4.048577
75%,1.000000,36.389135,4.679063
max,1.000000,40.874458,6.315182



=== Pack norms and best scale s (PACK ≈ s * NPZ) ===


,pack_norm,s_best
count,203.000000,203.000000
mean,35.824992,35.825003
std,2.414183,2.414240
min,27.797810,27.797695
25%,34.458364,34.458431
50%,35.591122,35.590664
75%,37.389133,37.389233
max,41.874470,41.874153



=== After compensation ===
• L2(PACK, s*NPZ)  -> should be ~0 if only a scale differs
• cos(NPZ, PACK_norm) -> should be ~1 if directions match


,"L2(PACK, s*NPZ)","cos(NPZ, PACK_norm)"
count,203.000000,203.000000
mean,0.007407,1.000000
std,0.000548,0.000000
min,0.005523,1.000000
25%,0.007095,1.000000
50%,0.007365,1.000000
75%,0.007770,1.000000
max,0.008892,1.000000



Worst 10 by cos after normalization:


,dataset,group,id,row,type,"cos(NPZ, PACK_norm)","L2(PACK, s*NPZ)",pack_norm,s_best,pack_path,npz_path,caption
0,Objaverse,000-073,07f183d28e0c408a92add93a084cf2e9,0,original,1.000000,0.007089,35.329811,35.329300,/mnt/disk1/souhail_data/objaverse-processed/me...,/mnt/disk2/souhail_data/Objaverse/000-073/07f1...,Amx10 RCR 1/72 model miniature
60,Objaverse,000-087,9bc913ff9e3145a3a51497ba72624eff,3,retrieval,1.000000,0.008413,39.858894,39.858631,/mnt/disk1/souhail_data/objaverse-processed/me...,/mnt/disk2/souhail_data/Objaverse/000-087/9bc9...,2560x1440 Mirrors Edge Cruzr Wallpaper 1920x1080
115,Objaverse,000-131,eeb73fc4feeb44ca91b843b579b4c38d,10,retrieval,1.000000,0.008284,41.196064,41.195923,/mnt/disk1/souhail_data/objaverse-processed/me...,/mnt/disk2/souhail_data/Objaverse/000-131/eeb7...,Haus 3D Model
91,Objaverse,000-061,c6ced88817194f7aade2230795eaa5f8,2,msft,1.000000,0.008098,36.618336,36.618179,/mnt/disk1/souhail_data/objaverse-processed/me...,/mnt/disk2/souhail_data/Objaverse/000-061/c6ce...,a tree with green leaves
199,Objaverse,000-041,b8d3eaa9239e47698e4b35a5de6d6016,11,retrieval,1.000000,0.007294,36.669537,36.669674,/mnt/disk1/souhail_data/objaverse-processed/me...,/mnt/disk2/souhail_data/Objaverse/000-041/b8d3...,mayan culture pyramid icon vector illustration...
174,Objaverse,000-020,8e9a9ce90141496980c7fa33e99740a1,3,retrieval,1.000000,0.007227,35.153049,35.152664,/mnt/disk1/souhail_data/objaverse-processed/me...,/mnt/disk2/souhail_data/Objaverse/000-020/8e9a...,Allosaurus Kit 13cm 10thick 3d printed
34,ShapeNet,04468005,bea62459ad724a7a3b371d842587b0d2,15,retrieval,1.000000,0.007777,37.378731,37.378845,/mnt/disk1/souhail_data/objaverse-processed/me...,/mnt/disk2/souhail_data/ShapeNet/04468005/bea6...,Drift Sword 3d printed
49,Objaverse,000-039,535455b539d0416d9605b0ddc5fb0f0c,11,retrieval,1.000000,0.007254,35.245434,35.245644,/mnt/disk1/souhail_data/objaverse-processed/me...,/mnt/disk2/souhail_data/Objaverse/000-039/5354...,Urbanlandscape25
67,Objaverse,000-087,9bc913ff9e3145a3a51497ba72624eff,10,retrieval,1.000000,0.007606,37.081573,37.081970,/mnt/disk1/souhail_data/objaverse-processed/me...,/mnt/disk2/souhail_data/Objaverse/000-087/9bc9...,Screenshot of obscurus Demo
81,Objaverse,000-148,1349d05138b844369f62b2e52dcfcb2f,7,retrieval,1.000000,0.008234,39.006920,39.008034,/mnt/disk1/souhail_data/objaverse-processed/me...,/mnt/disk2/souhail_data/Objaverse/000-148/1349...,Battle Droid Legs


In [4]:
import torch, open_clip

device = "cuda"
model, _, _ = open_clip.create_model_and_transforms(
    "ViT-bigG-14", pretrained="laion2b_s39b_b160k", device=device
)
model.eval()

# Ask the TEXT tower to also return per-token features
model.text.output_tokens = True

tokenizer = open_clip.get_tokenizer("ViT-bigG-14")
txt = tokenizer(["a photo of a red chair"]).to(device)

with torch.no_grad():
    # Returns: pooled global (projected) and last-layer token features
    g, toks = model.text(txt)          # g: [B, D]; toks: [B, L, d_model] or [B, L, D] depending on backend

# If tokens are in model-width (d_model) and you want them in the projected D space (to match images):
if getattr(model, "text_projection", None) is not None and toks.shape[-1] != g.shape[-1]:
    toks = toks @ model.text_projection    # [B, L, D]
toks = torch.nn.functional.normalize(toks, dim=-1)
g    = torch.nn.functional.normalize(g,    dim=-1)


AttributeError: 'CLIP' object has no attribute 'text'

In [5]:
model

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1664, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((1664,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-47): 48 x ResidualAttentionBlock(
          (ln_1): LayerNorm((1664,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1664, out_features=1664, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((1664,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1664, out_features=8192, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=8192, out_features=1664, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((1664,), eps=1e-05, elementwi

In [5]:
# === Compare API global vs "manual" global captured via ln_final hook (no attn_mask issues) ===
import numpy as np, torch, pandas as pd
import open_clip

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME  = "ViT-bigG-14"
PRETRAINED  = "laion2b_s39b_b160k"   # or a local checkpoint path

TEXTS = [
    "a red chair",
    "a small wooden table",
    "a futuristic electric car",
    "green potted plant on a desk",
    "a medieval iron sword with a leather handle",
]

@torch.no_grad()
def encode_text_api(model, token_ids: torch.Tensor):
    v = model.encode_text(token_ids)
    return torch.nn.functional.normalize(v.float(), dim=-1)

def cosine_sim(a, b, eps=1e-8):
    a = a.astype(np.float32); b = b.astype(np.float32)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + eps))

# --- load model/tokenizer
print(f"Loading {MODEL_NAME} / {PRETRAINED} on {DEVICE} …")
model, _, _ = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED, device=DEVICE)
model.eval().to(DEVICE)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)

# --- tokenize
tok = tokenizer(TEXTS).to(DEVICE)  # [B, L]

# --- hook ln_final to capture token features after transformer+LN
captured = {"tokens_C": None}  # tokens in model width C before projection

def _hook_ln_final(_mod, _inp, out):
    # out: [B, L, C]
    captured["tokens_C"] = out

hook = model.ln_final.register_forward_hook(_hook_ln_final)

with torch.no_grad():
    api = encode_text_api(model, tok)           # triggers ln_final, fills captured["tokens_C"]

hook.remove()

# --- reconstruct manual global from captured tokens
tokens_C = captured["tokens_C"]                       # [B, L, C]
assert tokens_C is not None, "ln_final hook did not fire."

# Project tokens to D (to match encode_text output space)
if getattr(model, "text_projection", None) is not None:
    tokens_D = tokens_C @ model.text_projection       # [B, L, D]
else:
    tokens_D = tokens_C

# Normalize tokens (optional; not necessary for global, but good to keep consistent)
tokens_D = torch.nn.functional.normalize(tokens_D.float(), dim=-1)

# EOT index per sample (same as encode_text)
eot = tok.argmax(dim=-1)                              # [B]
manual_global = tokens_D[torch.arange(tokens_D.shape[0]), eot, :]  # [B, D]

# Normalize manual global exactly like API
manual_global = torch.nn.functional.normalize(manual_global, dim=-1)

# --- compare
api_np = api.cpu().numpy()
man_np = manual_global.detach().cpu().numpy()

rows = []
for i, t in enumerate(TEXTS):
    a, m = api_np[i], man_np[i]
    rows.append({
        "i": i,
        "cos(man,api)": cosine_sim(m, a),
        "L2(man,api)" : float(np.linalg.norm(m - a)),
        "max|man-api|": float(np.max(np.abs(m - a))),
        "text": t if len(t) <= 120 else (t[:117] + "…"),
    })

df = pd.DataFrame(rows)
with pd.option_context('display.float_format', '{:.8f}'.format):
    print("\n=== Summary (manual via hook vs API global) ===")
    print(df[["cos(man,api)", "L2(man,api)", "max|man-api|"]].describe())

print("\nPer-text diffs (worst first):")
print(df.sort_values("cos(man,api)").head(len(TEXTS))[["i","cos(man,api)","L2(man,api)","max|man-api|","text"]].to_string(index=False))


Loading ViT-bigG-14 / laion2b_s39b_b160k on cuda …

=== Summary (manual via hook vs API global) ===
       cos(man,api)  L2(man,api)  max|man-api|
count    5.00000000   5.00000000    5.00000000
mean     1.00000005   0.00000034    0.00000005
std      0.00000007   0.00000002    0.00000001
min      1.00000000   0.00000031    0.00000004
25%      1.00000000   0.00000033    0.00000004
50%      1.00000000   0.00000034    0.00000004
75%      1.00000012   0.00000035    0.00000004
max      1.00000012   0.00000035    0.00000006

Per-text diffs (worst first):
 i  cos(man,api)  L2(man,api)  max|man-api|                                        text
 0           1.0 3.392900e-07  4.470348e-08                                 a red chair
 1           1.0 3.475211e-07  3.725290e-08                        a small wooden table
 2           1.0 3.114002e-07  5.960464e-08                   a futuristic electric car
 3           1.0 3.326784e-07  4.470348e-08                green potted plant on a desk
 4    

In [6]:
import torch
from typing import Dict

@torch.no_grad()
def text_features_with_tokens(model, token_ids: torch.Tensor, project_to_d: bool = True, normalize: bool = True) -> Dict[str, torch.Tensor]:
    """
    Works on open_clip.model.CLIP (no custom text tower needed).
    Returns a dict with:
      - global: [B, D] pooled at EOT
      - tokens: [B, L, D] per-token (projected to D if project_to_d=True)
      - mask:   [B, L]  True for valid (non-padding) positions
    """
    captured = {"tokens_C": None}

    def _hook_ln_final(_mod, _inp, out):
        # out: [B, L, C] after transformer + LN
        captured["tokens_C"] = out

    hook = model.ln_final.register_forward_hook(_hook_ln_final)

    # Run the usual API forward; this triggers the hook
    global_api = model.encode_text(token_ids)

    hook.remove()

    # Pull tokens at model width C
    x = captured["tokens_C"]                  # [B, L, C]
    assert x is not None, "ln_final hook didn't fire"

    # Project to D (same space as encode_text)
    if project_to_d and getattr(model, "text_projection", None) is not None:
        toks = x @ model.text_projection      # [B, L, D]
        glo  = x[torch.arange(x.shape[0]), token_ids.argmax(dim=-1), :] @ model.text_projection
    else:
        toks = x
        glo  = x[torch.arange(x.shape[0]), token_ids.argmax(dim=-1), :]

    # Build mask (padding tokens are zeros in tokenizer output)
    # open_clip tokenizer pads with 0; BOS=49406, EOT=49407 for BPE models
    mask = token_ids.ne(0)

    # Normalize to unit length (matches encode_text’s convention)
    glo  = torch.nn.functional.normalize(glo.float(),  dim=-1)
    toks = torch.nn.functional.normalize(toks.float(), dim=-1)

    return {"global": glo, "tokens": toks, "mask": mask, "global_api": torch.nn.functional.normalize(global_api.float(), dim=-1)}


In [7]:
import open_clip, torch, numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
model, _, _ = open_clip.create_model_and_transforms("ViT-bigG-14", pretrained="laion2b_s39b_b160k", device=device)
model.eval().to(device)
tokenizer = open_clip.get_tokenizer("ViT-bigG-14")

texts = ["a red chair", "a wooden table", "a futuristic car"]
tok = tokenizer(texts).to(device)

out = text_features_with_tokens(model, tok)
g_api = out["global_api"].cpu().numpy()  # [B, D]
g_man = out["global"].cpu().numpy()      # [B, D]
T     = out["tokens"].cpu().numpy()      # [B, L, D] per-token
mask  = out["mask"].cpu().numpy().astype(bool)

# quick sanity: cosine between manual and api globals
def cos(a,b): return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+1e-8))
print([cos(g_api[i], g_man[i]) for i in range(len(texts))])  # ~1.0 each


[1.0, 1.0000001192092896, 1.0]


In [ ]:
# === Validate text tokens: ln_final hook vs text tower output_tokens (when available) ===
import torch, numpy as np, pandas as pd

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME  = "ViT-bigG-14"
PRETRAINED  = "laion2b_s39b_b160k"

TEXTS = [
    "a red chair on a white floor",
    "a small wooden coffee table",
    "a futuristic electric car with sharp angles",
    "green potted plant on a desk",
    "a medieval iron sword with a leather handle",
    "two blue ceramic bowls stacked together",
]

def cosine_sim(a, b, eps=1e-8):
    a = a.astype(np.float32); b = b.astype(np.float32)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + eps))

# ---- Load base CLIP (API we use in embeddings.py)
import open_clip
print(f"Loading base CLIP: {MODEL_NAME}/{PRETRAINED} on {DEVICE} …")
model_base, _, _ = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED, device=DEVICE)
model_base.eval().to(DEVICE)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)

tok = tokenizer(TEXTS).to(DEVICE)      # [B, L]

# ---- Hook-based tokens (the exact method used in embeddings.py)
@torch.no_grad()
def tokens_via_hook(model, token_ids):
    captured = {"tokens_C": None}
    def _hook_ln_final(_m, _i, out): captured["tokens_C"] = out  # [B, L, C]
    h = model.ln_final.register_forward_hook(_hook_ln_final)
    api_g = model.encode_text(token_ids)  # triggers hook
    h.remove()
    xC = captured["tokens_C"]                              # [B, L, C]
    assert xC is not None, "ln_final hook did not fire"

    proj = getattr(model, "text_projection", None)
    assert proj is not None, "expected text_projection on this OpenCLIP text tower"
    xD = xC @ proj                                        # [B, L, D]
    xD = torch.nn.functional.normalize(xD.float(), dim=-1)
    api_g = torch.nn.functional.normalize(api_g.float(), dim=-1)
    return xD, api_g

tok_hook_D, api_global = tokens_via_hook(model_base, tok)     # [B, L, D], [B, D]
tok_hook_np = tok_hook_D.cpu().numpy()
api_g_np    = api_global.cpu().numpy()



# Extra invariants (should all be True)
pad_id, bos_id, eot_id = 0, 49406, 49407
tok_ids_np = tok.cpu().numpy()
B, L = tok_ids_np.shape

# (A) Each row's content length equals eot_index-1 (excluding BOS/EOT)
length_ok = all(int(tok_ids_np[i].argmax()) >= 1 for i in range(B))

# (B) Tokens are unit-normalized (within tiny epsilon)
norms_ok = np.allclose(
    np.linalg.norm(tok_hook_np, axis=-1)[tok_ids_np != 0], 1.0, atol=1e-3
)

# (C) Reconstructed global from EOT matches API global (already summarized)
eot_ok = all(
    abs(1.0 - np.dot(
        tok_hook_np[i, tok_ids_np[i].argmax()], api_g_np[i]
    ) / (np.linalg.norm(tok_hook_np[i, tok_ids_np[i].argmax()]) * np.linalg.norm(api_g_np[i]) + 1e-8)) < 1e-6
    for i in range(B)
)

print("length_ok:", length_ok, "| norms_ok:", norms_ok, "| eot_ok:", eot_ok)




# ---- Optionally: tokens via CustomTextCLIP (text tower with output_tokens=True)
def try_tokens_via_text_tower(model_name, pretrained, texts):
    try:
        # Some OpenCLIP versions accept custom_text= True; others expose force_custom_text.
        try:
            model_cust, _, _ = open_clip.create_model_and_transforms(model_name, pretrained=pretrained, device=DEVICE, custom_text=True)
        except TypeError:
            # Fallback signature for older versions (if available); if not, this will raise.
            model_cust, _, _ = open_clip.create_model_and_transforms(model_name, pretrained=pretrained, device=DEVICE)
        model_cust.eval().to(DEVICE)
        # Require output_tokens on text tower
        if not hasattr(model_cust, "text"):
            raise RuntimeError("No `model.text` submodule on this build.")
        if not hasattr(model_cust.text, "output_tokens"):
            raise RuntimeError("This text tower has no `output_tokens` flag.")
        model_cust.text.output_tokens = True

        tok2 = tokenizer(texts).to(DEVICE)
        with torch.no_grad():
            out = model_cust.text(tok2)
        # Expect either (global, tokens_C) or (tokens_C, global)
        if isinstance(out, (tuple, list)) and len(out) == 2:
            a, b = out
            if a.ndim == 2 and b.ndim == 3: g_c, tC = a, b
            elif a.ndim == 3 and b.ndim == 2: tC, g_c = a, b
            else:
                raise RuntimeError("Unexpected tuple shapes from text tower.")
        else:
            # Some versions may return tokens directly; but we need both
            raise RuntimeError("Text tower didn't return a (global, tokens) tuple.")
        # Project tokens to D if needed
        proj = getattr(model_cust.text, "text_projection", None) or getattr(model_cust, "text_projection", None)
        if proj is not None and tC.shape[-1] != g_c.shape[-1]:
            tD = tC @ proj
        else:
            tD = tC
        tD = torch.nn.functional.normalize(tD.float(), dim=-1)
        g_c = torch.nn.functional.normalize(g_c.float(), dim=-1)
        return tD.cpu().numpy(), g_c.cpu().numpy(), tok2
    except Exception as e:
        print("[NOTE] Could not validate against text tower directly:", repr(e))
        return None, None, None

tok_text_np, g_text_np, tok_text_ids = try_tokens_via_text_tower(MODEL_NAME, PRETRAINED, TEXTS)

# ---- Build masks and EOT indices (pad=0; we exclude BOS=49406 and EOT=49407)
pad_id, bos_id, eot_id = 0, 49406, 49407
tok_ids_np = tok.cpu().numpy()
B, L = tok_ids_np.shape
valid_slices = []
for i in range(B):
    eot = int(tok_ids_np[i].argmax())            # EOT index in this sample
    # slice tokens that correspond to actual text content: (1 .. eot-1)
    start = 1
    end = max(start, eot)                        # guard
    valid_slices.append(slice(start, end))

# ---- Checks
rows = []

# 1) Internal consistency: EOT row from hook tokens should reproduce API global
for i in range(B):
    eot = int(tok_ids_np[i].argmax())
    v_eot = tok_hook_np[i, eot, :]                     # [D]
    v_eot = v_eot / (np.linalg.norm(v_eot) + 1e-8)
    cos_eot_api = cosine_sim(v_eot, api_g_np[i])
    rows.append({"kind":"EOT_vs_API", "i": i, "cos": cos_eot_api})

# 2) If text tower tokens are available, compare token-by-token (excluding BOS/EOT)
if tok_text_np is not None:
    # ensure both batches are the same set/order of texts
    assert np.allclose(tok_text_ids.cpu().numpy(), tok_ids_np), "Tokenizer outputs differ between models"
    for i in range(B):
        s = valid_slices[i]
        A = tok_hook_np[i, s, :]                      # [Ti, D]
        Bc = tok_text_np[i, s, :]                     # [Ti, D]
        Ti = A.shape[0]
        if Ti == 0:
            rows.append({"kind":"per_token", "i": i, "min_cos": np.nan, "mean_cos": np.nan})
            continue
        # cosine per token
        cos_list = [cosine_sim(A[j], Bc[j]) for j in range(Ti)]
        rows.append({"kind":"per_token", "i": i, "min_cos": float(np.min(cos_list)), "mean_cos": float(np.mean(cos_list))})
        # also check EOT equality via tower globals
        eot = int(tok_ids_np[i].argmax())
        cos_eot_tower = cosine_sim(tok_hook_np[i, eot], g_text_np[i])
        rows.append({"kind":"EOT_vs_TOWER_G", "i": i, "cos": cos_eot_tower})

# Summarize
df = pd.DataFrame(rows)
# print("\n=== EOT token from hook should match API global ===")
# print(df[df.kind=="EOT_vs_API"].pivot_table(index=None, values="cos", aggfunc=["count","mean","min","max"]))

# if tok_text_np is not None:
#     print("\n=== Token-by-token cosine: hook tokens vs text tower tokens (exclude BOS/EOT) ===")
#     print(df[df.kind=="per_token"].pivot_table(index=None, values=["min_cos","mean_cos"], aggfunc=["count","mean","min","max"]))
#     print("\n=== EOT token vs text tower global ===")
#     print(df[df.kind=="EOT_vs_TOWER_G"].pivot_table(index=None, values="cos", aggfunc=["count","mean","min","max"]))
# else:
#     print("\n[INFO] Skipped direct text tower comparison (not available in this build). "
#           "EOT-vs-API check still guarantees token pipeline correctness.")
# After: df = pd.DataFrame(rows)

def summarize_series(name, s):
    if s.empty:
        print(f"\n=== {name} ===\n(no rows)")
    else:
        print(f"\n=== {name} ===")
        print(s.agg(['count','mean','min','max']))

# 1) EOT token vs API global
summarize_series(
    "EOT token from hook vs API global (cos)",
    df.loc[df.kind == "EOT_vs_API", "cos"]
)

# 2) Token-by-token (only if present — skip safely)
pt = df[df.kind == "per_token"]
if {"min_cos","mean_cos"}.issubset(pt.columns):
    print("\n=== Token-by-token cosine (hook vs text tower) ===")
    print("min_cos:")
    print(pt["min_cos"].agg(['count','mean','min','max']))
    print("\nmean_cos:")
    print(pt["mean_cos"].agg(['count','mean','min','max']))
else:
    print("\n=== Token-by-token cosine (hook vs text tower) ===\n(no rows: text tower not available)")

# 3) EOT vs text tower global (only if present)
summarize_series(
    "EOT token from hook vs text tower global (cos)",
    df.loc[df.kind == "EOT_vs_TOWER_G", "cos"] if "cos" in df.columns else pd.Series([], dtype=float)
)


Loading base CLIP: ViT-bigG-14/laion2b_s39b_b160k on cuda …
length_ok: True | norms_ok: True | eot_ok: True
[NOTE] Could not validate against text tower directly: RuntimeError('No `model.text` submodule on this build.')

=== EOT token from hook vs API global (cos) ===
count    6.0
mean     1.0
min      1.0
max      1.0
Name: cos, dtype: float64

=== Token-by-token cosine (hook vs text tower) ===
(no rows: text tower not available)

=== EOT token from hook vs text tower global (cos) ===
(no rows)


: 